# 02 — Prototype to package: the handoff

The Ridge baseline (notebook 01) left accuracy on the table. Here we prototype an
XGBoost model ad-hoc, then **re-express the exact same experiment through the
`mlops_lab` package + YAML config** — the moment a personal experiment becomes a
team-owned, CI-tested model candidate.

Tracking goes to the Azure ML workspace (`export MLFLOW_TRACKING_URI=$(make -s azure-uri)`),
or a local dev server (`make mlflow-up`) when offline.

In [ ]:
import os

os.environ.setdefault("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")

import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

bunch = fetch_california_housing(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    bunch.data, bunch.target, test_size=0.2, random_state=42
)

## Ad-hoc prototype

Params copied by hand, metrics computed by hand — the usual prototype mess.

In [ ]:
params = dict(
    n_estimators=300, max_depth=7, learning_rate=0.05,
    tree_method="hist", random_state=42, n_jobs=2,
)
proto = XGBRegressor(**params)
proto.fit(X_train, y_train)
pred = proto.predict(X_test)

adhoc = {
    "rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
    "r2": float(r2_score(y_test, pred)),
}
adhoc

## The handoff: same experiment, through the package

Everything above is already encoded in `configs/housing_xgb.yaml` — dataset, split
seed, params, plus the parts the notebook *doesn't* have: quality gates and the
champion-comparison policy. `run_training` is the exact code CI runs.

In [ ]:
from mlops_lab.config import load_config
from mlops_lab.train import run_training

cfg = load_config("../configs/housing_xgb.yaml")
print(cfg.model.params)
result = run_training(cfg)
result.metrics

In [ ]:
# Same seed, same params, same data -> the package reproduces the prototype.
assert abs(adhoc["rmse"] - result.metrics["rmse"]) < 1e-6, (adhoc, result.metrics)
print("package reproduces the ad-hoc prototype exactly \u2705")

In [ ]:
# And the gates CI will enforce:
from mlops_lab.evaluate import check_gates

gate = check_gates(result.metrics, cfg.gates)
print("gates passed:", gate.passed, gate.failures or "")

## From here, CI owns it

The data scientist's job ends with a pull request:

1. **Edit the YAML** (`configs/housing_xgb.yaml`) — new params, new gates, or a new
   config file for a new candidate.
2. **Open a PR.** CI lints, unit-tests, trains the candidate, checks the absolute
   quality gates, compares it against the current production champion in the Azure ML
   Model Registry, and posts the metrics table on the PR.
3. **Merge.** CD retrains, registers the best gate-passing candidate in the Azure ML
   Model Registry, and tags it `stage=staging` (see it in Studio → Models).
4. **A manager dispatches the Deploy workflow** — staging deploys automatically;
   the production step waits for their explicit approval, then the model is tagged
   `stage=production` and (mock-)deployed.

No model reaches production from a laptop.